# 04 — Kiểm chứng tập mẫu 40 kết quả MIMIC-IV

**Nhiệm vụ Jira SPP-62:** Kiểm tra danh sách cố định 40 `labevent_id` ở **Mục 6.3, trang 16–18** của *Báo cáo khảo sát MIMIC-IV LIS FHIR* (13/10/2026); đối chiếu tiêu chí ở Mục 4.6 và 6.1–6.2 với *Kế hoạch thực nghiệm v1.1, Mục 5.2*.

- **Không lấy mẫu ngẫu nhiên** và không sinh thêm kết quả giả. Danh sách kỳ vọng bên dưới được **chép từ báo cáo**; notebook đối chiếu nó với CSV gốc, rồi **tự chọn lại 40 dòng từ CSV gốc** theo [`docs/quy_tac_chon_mau_40.md`](../docs/quy_tac_chon_mau_40.md) và so với danh sách trong báo cáo (phải ra 40/40).
- `PASS` nghĩa là thông tin trong báo cáo khớp CSV và điều kiện kiểm. Quy tắc tự chọn được **tìm lại từ danh sách đã chốt**, nên khớp 40/40 **không chứng minh** đây là cách duy nhất hay tối ưu, và cũng không chứng minh prototype đã hoạt động.
- Cần dùng CSV MIMIC-IV Demo **2.2** gốc, không dùng fixture synthetic. Kết quả xuất vào `reports/`.
- Bắt đầu với **Run All** sau khi sửa đường dẫn nếu cần. Nếu thiếu dữ liệu, notebook sẽ báo lỗi rõ ràng thay vì tự chuyển sang dữ liệu mẫu.

In [ ]:
from pathlib import Path
from io import StringIO
from decimal import Decimal
import hashlib
import pandas as pd
from IPython.display import display
from analysis_core.selection import CORE_ITEMIDS, clean, dec, screen_labevents, select_core_40

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Hãy mở notebook từ thư mục repo có pyproject.toml")
REPORTS = ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
RAW_DIR = ROOT / "data" / "raw"

# CHỈNH 3 ĐƯỜNG DẪN NÀY NẾU FILE CSV THỰC NẰM Ở THƯ MỤC KHÁC (vd. raw/hosp).
PATHS = {
    "patients": RAW_DIR / "patients.csv.gz",
    "labevents": RAW_DIR / "labevents.csv.gz",
    "d_labitems": RAW_DIR / "d_labitems.csv.gz",
}
for name, path in PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy {name}: {path}. Hãy sửa PATHS trong cell đầu tiên.")
print("ROOT:", ROOT)
print("CSV:", {name: str(p) for name, p in PATHS.items()})

## 1. Bảng kỳ vọng 40 kết quả (nguồn: báo cáo, Mục 6.3)

Đây là **nội dung AI đã báo cáo**, chưa được xem là dữ liệu đúng cho đến khi đối chiếu với `labevents.csv.gz` và `d_labitems.csv.gz`. Số `csv_record` trong tài liệu tính từ 2 (dòng header là 1).

In [ ]:
EXPECTED_40_CSV = 'labevent_id,subject_id,itemid,label,fluid,category,charttime,valuenum,valueuom,ref_range_lower,ref_range_upper,csv_record\n174,10000032,50971,Potassium,Blood,Chemistry,2180-06-03 12:00:00,3.4,mEq/L,3.3,5.1,5671\n190,10000032,50971,Potassium,Blood,Chemistry,2180-06-03 12:00:00,3.3,mEq/L,3.3,5.1,5668\n8978,10001217,50971,Potassium,Blood,Chemistry,2157-11-18 18:30:00,4.4,mEq/L,3.3,5.1,60523\n12639,10001725,50971,Potassium,Blood,Chemistry,2110-04-08 11:11:00,4.1,mEq/L,3.3,5.1,29545\n33,10000032,50971,Potassium,Blood,Chemistry,2180-03-23 11:51:00,3.7,mEq/L,3.3,5.1,98003\n106,10000032,50971,Potassium,Blood,Chemistry,2180-05-06 22:25:00,3.4,mEq/L,3.3,5.1,53529\n9033,10001217,50971,Potassium,Blood,Chemistry,2157-11-20 13:30:00,4.2,mEq/L,3.3,5.1,61855\n12650,10001725,50971,Potassium,Blood,Chemistry,2110-04-11 18:02:00,3.9,mEq/L,3.3,5.1,14906\n171,10000032,50912,Creatinine,Blood,Chemistry,2180-06-03 12:00:00,0.4,mg/dL,0.4,1.1,5660\n184,10000032,50912,Creatinine,Blood,Chemistry,2180-06-03 12:00:00,0.4,mg/dL,0.4,1.1,5677\n8972,10001217,50912,Creatinine,Blood,Chemistry,2157-11-18 18:30:00,0.6,mg/dL,0.4,1.1,60517\n12645,10001725,50912,Creatinine,Blood,Chemistry,2110-04-11 18:02:00,0.8,mg/dL,0.4,1.1,14910\n19,10000032,50912,Creatinine,Blood,Chemistry,2180-03-23 11:51:00,0.4,mg/dL,0.4,1.1,97991\n102,10000032,50912,Creatinine,Blood,Chemistry,2180-05-06 22:25:00,0.3,mg/dL,0.4,1.1,53531\n9029,10001217,50912,Creatinine,Blood,Chemistry,2157-11-20 13:30:00,0.6,mg/dL,0.4,1.1,61858\n12674,10001725,50912,Creatinine,Blood,Chemistry,2110-04-12 02:59:00,0.8,mg/dL,0.4,1.1,107341\n64,10000032,51222,Hemoglobin,Blood,Hematology,2180-03-23 11:51:00,14.9,g/dL,12,16,97961\n8949,10001217,51222,Hemoglobin,Blood,Hematology,2157-11-18 18:30:00,14.1,g/dL,12,16,60545\n12659,10001725,51222,Hemoglobin,Blood,Hematology,2110-04-11 18:02:00,13.9,g/dL,12,16,14888\n79,10000032,51222,Hemoglobin,Blood,Hematology,2180-05-06 22:25:00,14.3,g/dL,12,16,53544\n9014,10001217,51222,Hemoglobin,Blood,Hematology,2157-11-20 13:30:00,12.5,g/dL,12,16,61864\n12685,10001725,51222,Hemoglobin,Blood,Hematology,2110-04-12 02:59:00,12.6,g/dL,12,16,107321\n134,10000032,51222,Hemoglobin,Blood,Hematology,2180-05-07 05:05:00,12.7,g/dL,12,16,21219\n196,10000032,51222,Hemoglobin,Blood,Hematology,2180-06-22 11:15:00,14,g/dL,12,16,37203\n28240,10002428,50802,Base Excess,Blood,Blood Gas,2156-05-11 11:44:00,0,mEq/L,,,84323\n28277,10002428,50802,Base Excess,Blood,Blood Gas,2156-05-11 12:42:00,-2,mEq/L,,,82175\n32485,10002495,50802,Base Excess,Blood,Blood Gas,2141-05-24 10:10:00,-4,mEq/L,,,25585\n38784,10002930,50802,Base Excess,Blood,Blood Gas,2196-04-14 09:05:00,-9,mEq/L,,,20780\n27477,10002428,50802,Base Excess,Blood,Blood Gas,2156-04-19 15:44:00,1,mEq/L,,,84336\n27496,10002428,50802,Base Excess,Blood,Blood Gas,2156-04-19 19:41:00,1,mEq/L,,,81069\n32546,10002495,50802,Base Excess,Blood,Blood Gas,2141-05-26 00:26:00,-4,mEq/L,,,54071\n27534,10002428,50802,Base Excess,Blood,Blood Gas,2156-04-20 00:46:00,3,mEq/L,,,89819\n39724,10002930,51492,Protein,Urine,Hematology,2201-03-01 09:29:00,30,mg/dL,0,0,57167\n118,10000032,51492,Protein,Urine,Hematology,2180-05-07 00:10:00,30,mg/dL,,,28917\n27097,10002428,51492,Protein,Urine,Hematology,2155-11-10 12:36:00,30,mg/dL,,,72634\n28267,10002428,51492,Protein,Urine,Hematology,2156-05-11 12:30:00,100,mg/dL,,,60946\n38768,10002930,51492,Protein,Urine,Hematology,2196-04-14 09:00:00,30,mg/dL,,,10690\n38910,10002930,51492,Protein,Urine,Hematology,2197-04-07 04:00:00,30,mg/dL,,,12001\n27632,10002428,51492,Protein,Urine,Hematology,2156-04-20 12:47:00,30,mg/dL,,,90917\n27720,10002428,51492,Protein,Urine,Hematology,2156-04-21 16:46:00,30,mg/dL,,,90195\n'
expected = pd.read_csv(StringIO(EXPECTED_40_CSV), dtype="string", keep_default_na=False)
assert len(expected) == 40
assert expected["labevent_id"].nunique() == 40
print("Số kết quả trích từ báo cáo:", len(expected))
display(expected.groupby(["itemid", "label", "fluid", "category", "valueuom"], sort=False).size().rename("so_ket_qua").reset_index())
display(expected.head(5))

## 2. Đọc CSV gốc và xác nhận phiên bản

Đọc **dưới dạng chuỗi** để không làm mất `N/A`, `___`, dấu so sánh và số 0; kiểm SHA-256 của **file nén** theo báo cáo (Mục 1.1). Không đưa raw CSV lên GitHub.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

HASHES_FROM_REPORT = {
    "patients": "59dcc6cd8450ddf03ad32cc7d76e41caa4c9633541715987f590c74835cf6fae",
    "labevents": "d51a4cf1ea63245abe7d544b0d50a928e9ee2ec5eb9a5ce85500aed0bd6c6dfc",
    "d_labitems": "72ea5f020469fd24543bd98ab3bd0a4f645a5a4b7802d9d52af20d736f9d76db",
}
sha = pd.DataFrame([{
    "table": name, "sha256_matches_report": file_sha256(path) == HASHES_FROM_REPORT[name]
    if path.name.endswith(".csv.gz") else pd.NA,
    "comment": "SHA-256 của file nén" if path.name.endswith(".csv.gz") else "Không kiểm hash nén khi dùng CSV giải nén",
} for name, path in PATHS.items()])
display(sha)
if (sha["sha256_matches_report"] == False).any():
    print("⚠️ Có hash không khớp. Phải xác nhận nguồn trước khi kết luận.")

def read_raw(path):
    return pd.read_csv(path, dtype="string", keep_default_na=False, na_filter=False, low_memory=False)

patients = read_raw(PATHS["patients"])
labevents = read_raw(PATHS["labevents"])
d_labitems = read_raw(PATHS["d_labitems"])
for name, frame in (("patients", patients), ("labevents", labevents), ("d_labitems", d_labitems)):
    print(name, "rows:", len(frame), "columns:", len(frame.columns))
# pandas giữ thứ tự bản ghi; vị trí dòng đầu là số 2 vì dòng 1 là header.
labevents = labevents.copy()
labevents["_csv_record"] = pd.Series(range(2, len(labevents) + 2), dtype="Int64")
REQUIRED = {"patients": {"subject_id"}, "d_labitems": {"itemid", "label", "fluid", "category"},
            "labevents": {"labevent_id", "subject_id", "itemid", "charttime", "value", "valuenum",
                          "valueuom", "ref_range_lower", "ref_range_upper", "comments"}}
for name, df in (("patients",patients),("labevents",labevents),("d_labitems",d_labitems)):
    missing = REQUIRED[name] - set(df.columns)
    if missing:
        raise ValueError(f"CSV {name} thiếu các cột: {sorted(missing)}")

## 3. Kiểm điều kiện sàng lọc cơ bản (Mục 4.6)

Dùng **lý do không đạt đầu tiên** để tránh đếm một dòng vào nhiều nhóm loại. Khoảng tham chiếu thiếu được giữ thiếu, không tự điền `0`. `hadm_id` và `storetime` không bắt buộc. Ở đây, việc kiểm tra chỉ dựa trên dữ liệu nguồn, không liên quan đến việc hệ thống chạy thành công hay chưa.

Hàm sàng lọc nằm trong `analysis_core.selection` (`basic_screen_reason`, `screen_labevents`) và dùng chung với notebook 05.

In [ ]:
labevents["basic_screen"] = screen_labevents(labevents, patients, d_labitems)
basic_summary = (labevents["basic_screen"].value_counts().rename_axis("ket_qua")
                 .reset_index(name="so_dong"))
display(basic_summary)
print("Qua sàng lọc cơ bản:", int((labevents["basic_screen"] == "PASS").sum()), "(báo cáo: 82.289)")
print("Không qua sàng lọc:", int((labevents["basic_screen"] != "PASS").sum()), "(báo cáo: 25.438)")
print("Lưu ý: nếu số khác, cần đối chiếu thứ tự và cách chuẩn hóa của từng quy tắc; chưa tự kết luận CSV sai.")

## 4. Tự chọn 40 dòng từ dữ liệu gốc

Chạy hàm `select_core_40` (quy tắc ở [`docs/quy_tac_chon_mau_40.md`](../docs/quy_tac_chon_mau_40.md)) trên các dòng đạt sàng lọc của 5 loại xét nghiệm, **không dùng danh sách trong báo cáo làm đầu vào**. Sau đó so với danh sách ở mục 1.

Các bước: (1) lấy dòng ưu tiên, (2) chọn 3 bệnh nhân, (3) mỗi bệnh nhân lấy 2 kết quả sớm nhất, (4) bổ sung cho đủ 8 theo `labevent_id` nhỏ nhất còn lại. Cột `buoc` cho biết dòng được chọn ở bước nào.

Lưu ý: quy tắc này được tìm lại từ danh sách đã chốt, nên khớp 40/40 là điều kiện cần của notebook, không phải bằng chứng rằng báo cáo tự suy ra được danh sách.

In [ ]:
core_pool = labevents[(labevents["basic_screen"] == "PASS") & labevents["itemid"].isin(CORE_ITEMIDS)]
units_per_item = core_pool.groupby("itemid")["valueuom"].nunique()
assert (units_per_item == 1).all(), f"Có loại xét nghiệm nhiều hơn một đơn vị: {units_per_item.to_dict()}"
print("Số dòng đưa vào quy tắc:", len(core_pool), "| mỗi loại đúng 1 đơn vị:", bool((units_per_item == 1).all()))

selected_40 = select_core_40(core_pool)
got_ids, exp_ids = set(selected_40["labevent_id"]), set(expected["labevent_id"])
selected_40["trong_bao_cao"] = selected_40["labevent_id"].isin(exp_ids)

rule_check = (selected_40.groupby("itemid", sort=False)
              .agg(so_dong=("labevent_id", "size"), so_benh_nhan=("subject_id", "nunique"),
                   khop_bao_cao=("trong_bao_cao", "sum")).reset_index())
display(rule_check)
display(selected_40.groupby("buoc").size().rename("so_dong").reset_index())
print("Dòng không phải 'sớm nhất của bệnh nhân' (bước 1 và 4):")
display(selected_40[selected_40["buoc"] != 3][["itemid", "labevent_id", "subject_id", "charttime", "buoc", "ly_do"]])

n_rule_match = len(got_ids & exp_ids)
print(f"Quy tắc tự chọn khớp danh sách báo cáo: {n_rule_match}/{len(exp_ids)}")
print("Chỉ có trong báo cáo:", sorted(exp_ids - got_ids, key=int))
print("Chỉ có trong kết quả quy tắc:", sorted(got_ids - exp_ids, key=int))
assert got_ids == exp_ids, "Quy tắc chọn 40 dòng không ra đúng danh sách trong báo cáo"

## 5. Đối chiếu từng dòng trong danh sách 40

Đối chiếu `subject_id`, `itemid`, nhãn và nhóm xét nghiệm, `charttime`, giá trị số, đơn vị, hai cận, số thứ tự bản ghi và tiêu chí sàng lọc. Bất kỳ sai lệch nào sẽ được ghi rõ vào `failed_checks`.

In [ ]:
def same_decimal(a, b):
    x, y = dec(a), dec(b)
    return x is not None and y is not None and x == y

def same_optional_decimal(a, b):
    if clean(a) == "" or clean(b) == "":
        return clean(a) == clean(b)
    return same_decimal(a, b)

by_event = labevents.groupby("labevent_id", sort=False)
by_item = d_labitems.groupby("itemid", sort=False)
audit_rows = []
for _, exp in expected.iterrows():
    id_ = exp["labevent_id"]
    errors = []
    found = id_ in by_event.indices
    actual = None
    if not found:
        errors.append("khong_tim_thay_labevent_id")
    else:
        hits = by_event.get_group(id_)
        if len(hits) != 1:
            errors.append(f"labevent_id_khong_duy_nhat:{len(hits)}")
        else:
            actual = hits.iloc[0]
            for fld in ("subject_id", "itemid", "charttime", "valueuom"):
                if clean(exp[fld]) != clean(actual[fld]):
                    errors.append(f"sai_{fld}")
            if not same_decimal(exp["valuenum"], actual["value"]):
                errors.append("sai_value_so_voi_bao_cao")
            if not same_decimal(exp["valuenum"], actual["valuenum"]):
                errors.append("sai_valuenum_so_voi_bao_cao")
            for fld in ("ref_range_lower", "ref_range_upper"):
                if not same_optional_decimal(exp[fld], actual[fld]):
                    errors.append(f"sai_{fld}")
            if int(exp["csv_record"]) != int(actual["_csv_record"]):
                errors.append("sai_thu_tu_ban_ghi_csv")
            if actual["basic_screen"] != "PASS":
                errors.append("khong_qua_sang_loc:" + str(actual["basic_screen"]))
            it = actual["itemid"]
            if it not in by_item.indices or len(by_item.get_group(it)) != 1:
                errors.append("itemid_thieu_hoac_khong_duy_nhat_trong_d_labitems")
            else:
                cat = by_item.get_group(it).iloc[0]
                for fld in ("label", "fluid", "category"):
                    if clean(cat[fld]) != clean(exp[fld]):
                        errors.append(f"sai_danh_muc_{fld}")
    audit_rows.append({
        "labevent_id": id_, "itemid": exp["itemid"], "expected_subject_id": exp["subject_id"],
        "actual_subject_id": clean(actual["subject_id"]) if actual is not None else "",
        "expected_value": exp["valuenum"],
        "actual_value": clean(actual["value"]) if actual is not None else "",
        "expected_unit": exp["valueuom"],
        "actual_unit": clean(actual["valueuom"]) if actual is not None else "",
        "source_match": "PASS" if found and actual is not None else "FAIL",
        "basic_eligibility": "PASS" if actual is not None and actual["basic_screen"] == "PASS" else "FAIL",
        "status": "PASS" if not errors else "FAIL",
        "failed_checks": "; ".join(errors),
    })
audit = pd.DataFrame(audit_rows)
display(audit)
print("Tổng số khớp hoàn toàn:", int((audit.status == "PASS").sum()), "/", len(audit))

## 6. Kiểm tiêu chí bao phủ và các ca đặc biệt (Mục 6.1–6.2)

Tính các thống kê từ **bản ghi nguồn được truy xuất**, không chỉ từ bảng 40 dòng kỳ vọng. Đây là kiểm chứng bao phủ của mẫu, **không phải chứng minh** phương pháp chọn được xác định duy nhất.

In [ ]:
chosen = labevents[labevents["labevent_id"].isin(expected["labevent_id"])].copy()
chosen["_number"] = chosen["valuenum"].map(dec)
chosen["_low_missing"] = chosen["ref_range_lower"].map(clean).eq("")
chosen["_high_missing"] = chosen["ref_range_upper"].map(clean).eq("")

counts = chosen.groupby("itemid").agg(
    so_ket_qua=("labevent_id", "size"), so_benh_nhan=("subject_id", "nunique"),
    so_gia_tri=("_number", "nunique")
).reset_index()
display(counts)

coverage = {
    "40_ket_qua": len(chosen) == 40,
    "40_labevent_id_khac_nhau": chosen["labevent_id"].nunique() == 40,
    "5_itemid": chosen["itemid"].nunique() == 5,
    "moi_itemid_co_8_ket_qua": len(counts) == 5 and counts["so_ket_qua"].eq(8).all(),
    "moi_itemid_co_it_nhat_3_benh_nhan": len(counts) == 5 and counts["so_benh_nhan"].ge(3).all(),
    "6_benh_nhan_toan_tap": chosen["subject_id"].nunique() == 6,
    "25_ket_qua_du_hai_can": int((~chosen["_low_missing"] & ~chosen["_high_missing"]).sum()) == 25,
    "15_ket_qua_thieu_ca_hai_can": int((chosen["_low_missing"] & chosen["_high_missing"]).sum()) == 15,
    "0_ket_qua_chi_mot_can": int((chosen["_low_missing"] ^ chosen["_high_missing"]).sum()) == 0,
    "1_ket_qua_bang_0": sum(x == 0 for x in chosen["_number"] if x is not None) == 1,
    "4_ket_qua_am": sum(x < 0 for x in chosen["_number"] if x is not None) == 4,
}

def pair_exists(frame, require_equal):
    for _, grp in frame.groupby(["subject_id", "charttime"]):
        if grp["labevent_id"].nunique() >= 2:
            distinct = grp["_number"].nunique()
            if (distinct == 1 if require_equal else distinct >= 2):
                return True
    return False

coverage.update({
    "Potassium_cung_nguoi_cung_gio_khac_gia_tri": pair_exists(chosen[chosen.itemid == "50971"], False),
    "Creatinine_cung_nguoi_cung_gio_cung_gia_tri_khac_id": pair_exists(chosen[chosen.itemid == "50912"], True),
    "Hemoglobin_8_gia_tri_khac_nhau": chosen[chosen.itemid == "51222"]["_number"].nunique() == 8,
    "Base_Excess_co_0_va_so_am": (chosen[chosen.itemid == "50802"]["_number"].eq(Decimal(0)).any()
                                   and any(x < 0 for x in chosen[chosen.itemid == "50802"]["_number"] if x is not None)),
    "Protein_Urine_co_can_0_0_va_thieu_hai_can": (
        any(dec(r["ref_range_lower"]) == 0 and dec(r["ref_range_upper"]) == 0
            for _, r in chosen[chosen.itemid == "51492"].iterrows())
        and (chosen[chosen.itemid == "51492"]["_low_missing"] & chosen[chosen.itemid == "51492"]["_high_missing"]).any()),
})
coverage_df = pd.DataFrame([{"kiem_tra": name, "ket_qua": "PASS" if flag else "FAIL"}
                            for name, flag in coverage.items()])
display(coverage_df)
print("Số tiêu chí bao phủ đạt:", int(coverage_df.ket_qua.eq("PASS").sum()), "/", len(coverage_df))

## 7. Thống kê lý do không chọn (phân biệt loại và không nằm trong mẫu)

Theo báo cáo: 82.289 qua sàng lọc cơ bản; 2.763 thuộc `itemid` đa đơn vị cần rà soát; 79.526 thuộc `itemid` một đơn vị; trong đó chọn 40. Cách xác định **một đơn vị** cần nhất quán (ở đây xét trên những dòng qua sàng lọc cơ bản; nếu khác báo cáo cần kiểm tra cách tính). Những dòng còn lại *không nằm trong manifest* **không đồng nghĩa** dữ liệu sai.

In [ ]:
eligible = labevents[labevents["basic_screen"] == "PASS"].copy()
unit_counts = eligible.assign(_unit=eligible["valueuom"].map(clean)).groupby("itemid")["_unit"].nunique()
multi_unit_itemids = set(unit_counts[unit_counts > 1].index)
eligible_multi = int(eligible["itemid"].isin(multi_unit_itemids).sum())
eligible_single = len(eligible) - eligible_multi
print("Qua sàng lọc:", len(eligible), "(báo cáo 82.289)")
print("Trong nhóm có itemid đa đơn vị:", eligible_multi, "(báo cáo 2.763; xem lưu ý phương pháp)")
print("Thuộc itemid một đơn vị:", eligible_single, "(báo cáo 79.526)")
print("Thuộc itemid một đơn vị, không chọn vào 40:", eligible_single - len(chosen))
if eligible_multi != 2763:
    print("⚠️ Phân nhóm đa đơn vị khác báo cáo. Cần kiểm tra định nghĩa số đơn vị theo itemid.")

## 8. Xuất bằng chứng và hướng kết luận

Bảng này hỗ trợ giao cho thành viên khác kiểm chéo. `status=PASS` chỉ có nghĩa dòng khớp dữ liệu gốc theo các quy tắc đã lập trình; vẫn phải nhận xét thủ công việc chọn các dòng cụ thể. **Không dùng kết quả này để khẳng định toàn bộ hệ thống LIS → FHIR → EMR đã kiểm chứng.**

In [ ]:
audit_path = REPORTS / "sample_selection_audit_40.csv"
coverage_path = REPORTS / "sample_selection_coverage.csv"
manifest_path = REPORTS / "sample_selection_manifest_expected.csv"
rule_path = REPORTS / "sample_selection_core40_from_rule.csv"
summary_path = REPORTS / "sample_selection_audit_summary.md"
audit.to_csv(audit_path, index=False, encoding="utf-8-sig")
coverage_df.to_csv(coverage_path, index=False, encoding="utf-8-sig")
expected.to_csv(manifest_path, index=False, encoding="utf-8-sig")
selected_40.to_csv(rule_path, index=False, encoding="utf-8-sig")

n_pass = int(audit.status.eq("PASS").sum())
n_coverage = int(coverage_df.ket_qua.eq("PASS").sum())
summary_md = f"""# Kiểm chứng tập mẫu 40 kết quả — SPP-62

- **Nguồn**: MIMIC-IV Clinical Database Demo 2.2 (cần xác nhận checksum).
- **Danh sách kỳ vọng**: Báo cáo khảo sát, Mục 6.3, trang 16–18.
- **Kiểm dòng**: {n_pass}/40 PASS.
- **Tự chọn 40 dòng bằng quy tắc** (`docs/quy_tac_chon_mau_40.md`): {n_rule_match}/40 khớp danh sách báo cáo.
- **Tiêu chí bao phủ**: {n_coverage}/{len(coverage_df)} PASS.
- **Tổng qua sàng lọc cơ bản**: {len(eligible)} (báo cáo: 82.289).
- **Phát biểu chưa thể chứng minh**: Mục 6.2 của báo cáo không đủ để tự suy ra 40 ID; quy tắc trong
  `docs/quy_tac_chon_mau_40.md` được tìm lại từ danh sách đã chốt, nên có thể còn cách khác cho cùng kết quả.
  Cần ghi rõ đây là chọn mẫu có chủ đích.
- **Bước tiếp theo**: Ngọc Tú đối chiếu độc lập ID 174, 171, 184, 28240, 28277, 39724 và 118;
  xác nhận lý do lựa chọn và ký nhận kiểm chéo trước khi Done.
"""
summary_path.write_text(summary_md, encoding="utf-8")
for p in (audit_path,coverage_path,manifest_path,rule_path,summary_path):
    print("Đã lưu:", p)
if n_pass == 40 and n_coverage == len(coverage_df) and n_rule_match == 40:
    print("✅ Danh sách 40 dòng khớp các phép kiểm đã triển khai. CẦN kiểm chéo thủ công trước khi Jira Done.")
else:
    print("⚠️ Có sai lệch. Xem 'failed_checks' trong audit_40 và tiêu chí FAIL trong coverage.")